In [ ]:
import pandas as pd

# 1. 경로 설정
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
excel_path = rf"{base_path}\운영DB 스키마 명세.xlsx"

# 2. 명세서 로드
schema_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')

# 3. 재무상태표 테이블만 필터링
target_schema = schema_score[schema_score['table_name'] == 'company_financial_statement']

# 4. 번역표(딕셔너리) 생성: 물리명(대문자) -> 논리명(한글)
bs_map = {str(k).upper(): v for k, v in zip(target_schema['column_name'], target_schema['logical_column_name']) if pd.notna(v)}

# 5. 번역표가 제대로 만들어졌는지 샘플 출력 (15개만 확인)
print("=== [STEP 1] 재무상태표(BS) 번역표 검증 ===")
for i, (k, v) in enumerate(bs_map.items()):
    if i < 15:
        print(f"물리명: '{k}'  --->  논리명: '{v}'")

=== [STEP 1] 재무상태표(BS) 번역표 검증 ===
물리명: 'ID'  --->  논리명: '재무제표 아이디'
물리명: 'COMPANY_ID'  --->  논리명: '회사 아이디'
물리명: 'CREATED_AT'  --->  논리명: '생성일자'
물리명: 'FS_ACCT_DT'  --->  논리명: '결산일'
물리명: 'FS_VAL1'  --->  논리명: '자산'
물리명: 'FS_VAL2'  --->  논리명: '유동자산'
물리명: 'FS_VAL3'  --->  논리명: '당좌자산'
물리명: 'FS_VAL4'  --->  논리명: '현금 및 현금성 자산'
물리명: 'FS_VAL5'  --->  논리명: '국고보조금'
물리명: 'FS_VAL6'  --->  논리명: '외화현금 및 현금성 자산'
물리명: 'FS_VAL7'  --->  논리명: '단기예금(단기금융상품)'
물리명: 'FS_VAL8'  --->  논리명: '국고보조금'
물리명: 'FS_VAL9'  --->  논리명: '사용이 제한된 단기예금'
물리명: 'FS_VAL10'  --->  논리명: '외화단기예금(단기금융상품)'
물리명: 'FS_VAL11'  --->  논리명: '단기매매증권'


In [14]:
# === [STEP 2] 원본 데이터 로드 및 한글 컬럼명 변환 ===

# 1. 재무상태표 원본 로드
bs_path = rf"{base_path}\output\PROD_FLOWSCORE\PUBLIC\COMPANY_FINANCIAL_STATEMENT.csv"
df_bs_raw = pd.read_csv(bs_path, encoding='utf-8-sig')
print(f"1. 원본 데이터 로드 완료: {df_bs_raw.shape[0]}행 x {df_bs_raw.shape[1]}열")

# 2. 대소문자 불일치 방지를 위해 원본 컬럼명을 모두 대문자로 통일
df_bs_raw.columns = [str(col).upper() for col in df_bs_raw.columns]

# 3. STEP 1의 번역표(bs_map)를 적용하여 이름표 교체
df_bs_labeled = df_bs_raw.rename(columns=bs_map)

# 4. 검증: 우리가 심사 모델링에 꼭 써야 할 핵심 지표들이 잘 변환되었는지 확인
check_cols = ['회사 아이디', '결산일', '자산', '유동자산', '부채', '자본']
found_cols = [c for c in check_cols if c in df_bs_labeled.columns]

print(f"\n2. 변환 성공한 핵심 한글 컬럼: {found_cols}")

# 5. 데이터 값 샘플 확인
if len(found_cols) > 0:
    print("\n3. 변환된 데이터 샘플 (상위 5행):")
    display(df_bs_labeled[found_cols].head())
else:
    print("\n❌ 변환 실패: 한글 컬럼이 생성되지 않았습니다. 매핑을 다시 확인해야 합니다.")

1. 원본 데이터 로드 완료: 3562행 x 392열

2. 변환 성공한 핵심 한글 컬럼: ['회사 아이디', '결산일', '자산', '유동자산', '부채', '자본']

3. 변환된 데이터 샘플 (상위 5행):


,회사 아이디,결산일,자산,유동자산,부채,자본
0,1598,2023-12-31,3742196.0,996540.0,2867353.0,874843.0
1,1598,2022-12-31,3969242.0,3619261.0,1350351.0,2618891.0
2,1592,2024-12-31,13902077.0,7061183.0,6016522.0,7885555.0
3,1591,2022-12-31,1433740.0,1423740.0,1638001.0,-204261.0
4,1152,2022-12-31,170777.0,160777.0,110035.0,60742.0


In [15]:
# === [STEP 3] 손익계산서(IS) 매핑 및 결측치 진단 (중복 방어 추가) ===

# 1. 손익계산서 번역표(is_map) 생성
target_schema_is = schema_score[schema_score['table_name'] == 'company_income_statement']
is_map = {str(k).upper(): v for k, v in zip(target_schema_is['column_name'], target_schema_is['logical_column_name']) if pd.notna(v)}

# 2. 손익계산서 원본 로드 및 이름표 교체
is_path = rf"{base_path}\output\PROD_FLOWSCORE\PUBLIC\COMPANY_INCOME_STATEMENT.csv"
df_is_raw = pd.read_csv(is_path, encoding='utf-8-sig')
df_is_raw.columns = [str(col).upper() for col in df_is_raw.columns]
df_is_labeled = df_is_raw.rename(columns=is_map)

# 3. 데이터 건강 진단 함수 정의 (중복 컬럼 제거 로직 포함!)
def check_data_health_safe(df, requirements, report_name):
    # ★ 핵심 방어 로직: 이름이 중복된 컬럼이 있다면 첫 번째 컬럼만 남깁니다.
    df_safe = df.loc[:, ~df.columns.duplicated(keep='first')]
    
    print(f"\n" + "="*55)
    print(f"📊 {report_name} 필수 지표 진단 (총 {len(df_safe)}행)")
    print("="*55)
    
    report_data = []
    for col in requirements:
        if col in df_safe.columns:
            total_rows = len(df_safe)
            null_count = df_safe[col].isnull().sum()
            zero_count = (df_safe[col] == 0).sum()
            
            # 유효 데이터율 계산
            valid_rate = 100 - ((null_count + zero_count) / total_rows * 100)
            
            report_data.append({
                '지표명': col,
                '결측치(Null)': f"{null_count}건",
                '0값(Zero)': f"{zero_count}건",
                '유효 데이터율': f"{valid_rate:.1f}%"
            })
        else:
            report_data.append({
                '지표명': col, '결측치(Null)': "매핑 실패", '0값(Zero)': "-", '유효 데이터율': "0.0%"
            })
            
    display(pd.DataFrame(report_data))

# 4. 필수 지표 리스트
bs_req = ['자산', '유동자산', '부채', '자본', '단기차입금', '장기차입금']
is_req = ['매출액', '매출총이익(손실)', '영업이익(손실)', '이자비용']

# 5. 진단 실행 (df_bs_labeled는 STEP 2에서 넘어온 변수)
check_data_health_safe(df_bs_labeled, bs_req, "[재무상태표 BS]")
check_data_health_safe(df_is_labeled, is_req, "[손익계산서 IS]")


📊 [재무상태표 BS] 필수 지표 진단 (총 3562행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,자산,34건,0건,99.0%
1,유동자산,36건,0건,99.0%
2,부채,51건,0건,98.6%
3,자본,35건,0건,99.0%
4,단기차입금,1275건,3건,64.1%
5,장기차입금,1452건,4건,59.1%



📊 [손익계산서 IS] 필수 지표 진단 (총 3561행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,매출액,113건,1건,96.8%
1,매출총이익(손실),111건,1건,96.9%
2,영업이익(손실),38건,1건,98.9%
3,이자비용,659건,2건,81.4%


In [16]:
# === [확장판] 매뉴얼 대비 전체 재무 지표 매핑 및 결측치 진단 ===

# 1. 매뉴얼에서 요구하는 상세 심사 지표 풀세트 정의
bs_manual_req = [
    '자산', '유동자산', '비유동자산', '현금 및 현금성 자산', '매출채권', '재고자산', 
    '부채', '유동부채', '매입채무', '단기차입금', '장기차입금', '사채',
    '자본', '자본금', '이익잉여금'
]

is_manual_req = [
    '매출액', '매출원가', '매출총이익(손실)', 
    '판매비와관리비', '급여', '대손상각비', 
    '영업이익(손실)', '영업외수익', '영업외비용', '이자비용', 
    '당기순이익(순손실)'
]

# 2. 전수 검사 실행 (이전 스텝에서 정의한 check_data_health_safe 함수 사용)
print("=== 🔎 심사 매뉴얼 전면 자동화를 위한 데이터 전수 검사 ===")
check_data_health_safe(df_bs_labeled, bs_manual_req, "[재무상태표 BS 상세]")
check_data_health_safe(df_is_labeled, is_manual_req, "[손익계산서 IS 상세]")

=== 🔎 심사 매뉴얼 전면 자동화를 위한 데이터 전수 검사 ===

📊 [재무상태표 BS 상세] 필수 지표 진단 (총 3562행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,자산,34건,0건,99.0%
1,유동자산,36건,0건,99.0%
2,비유동자산,111건,0건,96.9%
3,현금 및 현금성 자산,49건,0건,98.6%
4,매출채권,428건,4건,87.9%
5,재고자산,1028건,10건,70.9%
6,부채,51건,0건,98.6%
7,유동부채,56건,0건,98.4%
8,매입채무,962건,3건,72.9%
9,단기차입금,1275건,3건,64.1%



📊 [손익계산서 IS 상세] 필수 지표 진단 (총 3561행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,매출액,113건,1건,96.8%
1,매출원가,596건,7건,83.1%
2,매출총이익(손실),111건,1건,96.9%
3,판매비와관리비,34건,1건,99.0%
4,급여,238건,1건,93.3%
5,대손상각비,매핑 실패,-,0.0%
6,영업이익(손실),38건,1건,98.9%
7,영업외수익,146건,1건,95.9%
8,영업외비용,276건,1건,92.2%
9,이자비용,659건,2건,81.4%


In [17]:
# === [전수 검사] 276홀딩스 매뉴얼 100% 커버리지 진단 ===

# 1. 매뉴얼에서 도출한 모든 필수/보조 계정 리스트
bs_full_req = [
    '자산', '유동자산', '비유동자산', '현금 및 현금성 자산', 
    '매출채권', '재고자산', '상품', '제품', 
    '개발비', '무형자산', 
    '부채', '유동부채', '매입채무', '미지급비용', 
    '단기차입금', '장기차입금', '사채', 
    '자본', '자본금', '자본잉여금', '이익잉여금'
]

is_full_req = [
    '매출액', '매출원가', '매출총이익(손실)', 
    '판매비와관리비', '급여', '지급수수료', '광고선전비', '대손상각비', 
    '영업이익(손실)', '영업외수익', '영업외비용', '이자비용', 
    '당기순이익(순손실)'
]

# 2. 전수 검사 실행
print("=== 🔎 심사 매뉴얼 전면 자동화를 위한 데이터 전수 검사 ===")
check_data_health_safe(df_bs_labeled, bs_full_req, "[재무상태표 BS 풀세트]")
check_data_health_safe(df_is_labeled, is_full_req, "[손익계산서 IS 풀세트]")

=== 🔎 심사 매뉴얼 전면 자동화를 위한 데이터 전수 검사 ===

📊 [재무상태표 BS 풀세트] 필수 지표 진단 (총 3562행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,자산,34건,0건,99.0%
1,유동자산,36건,0건,99.0%
2,비유동자산,111건,0건,96.9%
3,현금 및 현금성 자산,49건,0건,98.6%
4,매출채권,428건,4건,87.9%
5,재고자산,1028건,10건,70.9%
6,상품,2329건,10건,34.3%
7,제품,2269건,13건,35.9%
8,개발비,2785건,9건,21.6%
9,무형자산,1224건,6건,65.5%



📊 [손익계산서 IS 풀세트] 필수 지표 진단 (총 3561행)


,지표명,결측치(Null),0값(Zero),유효 데이터율
0,매출액,113건,1건,96.8%
1,매출원가,596건,7건,83.1%
2,매출총이익(손실),111건,1건,96.9%
3,판매비와관리비,34건,1건,99.0%
4,급여,238건,1건,93.3%
5,지급수수료,68건,0건,98.1%
6,광고선전비,1205건,4건,66.0%
7,대손상각비,매핑 실패,-,0.0%
8,영업이익(손실),38건,1건,98.9%
9,영업외수익,146건,1건,95.9%


In [18]:
import pandas as pd
import os

# 1. 경로 설정
base_path = r"C:\Users\cozy1\Documents\평가모델 3"
path_score = rf"{base_path}\output\PROD_FLOWSCORE\PUBLIC"
path_point = rf"{base_path}\output\PROD_FLOWPOINT\PUBLIC"
path_pension = rf"{base_path}\output\PROD_FLOWPENSION\PUBLIC" # 펜션(연금) 데이터 폴더 가정

# 2. 명세서 로드 및 매핑 함수
excel_path = rf"{base_path}\운영DB 스키마 명세.xlsx"
schema_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')
schema_point = pd.read_excel(excel_path, sheet_name='PROD_FLOWPOINT')

# 만약 연금 스키마가 별도 시트에 있다면 추가 (여기서는 score 시트에 있다고 가정하거나 에러 우회)
try:
    schema_pension = pd.read_excel(excel_path, sheet_name='PROD_FLOWPENSION')
except:
    schema_pension = schema_score # 시트가 없을 경우의 임시 방어

def get_mapping(df_schema, table_name):
    target = df_schema[df_schema['table_name'] == table_name]
    return {str(k).upper(): v for k, v in zip(target['column_name'], target['logical_column_name']) if pd.notna(v)}

# 3. 진단할 비재무/대안 테이블 리스트 (테이블명, 파일경로, 스키마, 화면출력명)
tables_to_check = [
    ('company_overview', rf"{path_score}\COMPANY_OVERVIEW.csv", schema_score, "[기업개황 Overview]"),
    ('company_credit_grade', rf"{path_score}\COMPANY_CREDIT_GRADE.csv", schema_score, "[신용등급 Credit Grade]"),
    ('company_links', rf"{path_point}\COMPANY_LINKS.csv", schema_point, "[공급망 링크 Links]"),
    ('company_pension_employee_statistics', rf"{path_pension}\COMPANY_PENSION_EMPLOYEE_STATISTICS.csv", schema_pension, "[인력/연금 Pension]")
]

# 4. 진단 엔진
def check_non_financial_health():
    for table_name, file_path, schema_df, report_name in tables_to_check:
        if not os.path.exists(file_path):
            print(f"\n⚠️ {report_name} 파일을 찾을 수 없습니다: {file_path}")
            continue
            
        # 데이터 로드
        df_raw = pd.read_csv(file_path, encoding='utf-8-sig', low_memory=False)
        df_raw.columns = [str(col).upper() for col in df_raw.columns]
        
        # 매핑 적용
        mapping = get_mapping(schema_df, table_name)
        df_labeled = df_raw.rename(columns=mapping)
        df_safe = df_labeled.loc[:, ~df_labeled.columns.duplicated(keep='first')]
        
        print(f"\n" + "="*60)
        print(f"📊 {report_name} 데이터 현황 (총 {len(df_safe)}행)")
        print("="*60)
        
        report_data = []
        # 논리명(한글명)으로 변환된 컬럼들만 추려서 검사 (시스템 컬럼 _ 제외)
        korean_cols = [c for c in mapping.values() if c in df_safe.columns and not c.startswith('_')]
        
        # 컬럼이 너무 많을 수 있으므로 상위 10개 핵심 컬럼만 출력
        for col in korean_cols[:10]: 
            total_rows = len(df_safe)
            null_count = df_safe[col].isnull().sum()
            valid_rate = 100 - (null_count / total_rows * 100)
            
            report_data.append({
                '지표명': col,
                '결측치(Null)': f"{null_count}건",
                '유효 데이터율': f"{valid_rate:.1f}%"
            })
            
        if report_data:
            display(pd.DataFrame(report_data))
        else:
            print("❌ 매핑된 한글 컬럼을 찾을 수 없습니다.")

# 5. 실행
check_non_financial_health()


📊 [기업개황 Overview] 데이터 현황 (총 1753행)


,지표명,결측치(Null),유효 데이터율
0,기업 개요 아이디,0건,100.0%
1,회사 아이디,0건,100.0%
2,생성일자,0건,100.0%
3,기업명,0건,100.0%
4,기업형태를 포함한 기업명,0건,100.0%
5,영문 기업명,438건,75.0%
6,사업자번호,7건,99.6%
7,법인번호,6건,99.7%
8,대표자명,1건,99.9%
9,기업 유형,0건,100.0%



📊 [신용등급 Credit Grade] 데이터 현황 (총 9461행)


,지표명,결측치(Null),유효 데이터율
0,기업 신용등급이력 5개년 아이디,0건,100.0%
1,회사 아이디,0건,100.0%
2,생성일자,0건,100.0%
3,기업 신용등급,1건,100.0%
4,기업 신용등급명,1건,100.0%
5,기업 신용등급 설명,1건,100.0%
6,등급분류,1건,100.0%
7,평가일자,1건,100.0%
8,결산기준일자,1572건,83.4%



📊 [공급망 링크 Links] 데이터 현황 (총 25036행)


,지표명,결측치(Null),유효 데이터율
0,primary key,0건,100.0%
1,생성일자,0건,100.0%
2,수정일자,0건,100.0%
3,삭제일자,25036건,0.0%
4,companies ID,0건,100.0%
5,companies ID,0건,100.0%
6,금액,0건,100.0%
7,거래일자,0건,100.0%
8,종류,0건,100.0%
9,참조 테이블명,0건,100.0%



⚠️ [인력/연금 Pension] 파일을 찾을 수 없습니다: C:\Users\cozy1\Documents\평가모델 3\output\PROD_FLOWPENSION\PUBLIC\COMPANY_PENSION_EMPLOYEE_STATISTICS.csv


In [19]:
# 1. 기존 명세서 기반 매핑 로직 (LINKS)
link_target = schema_point[schema_point['table_name'] == 'company_links']
link_map = {str(k).upper(): v for k, v in zip(link_target['column_name'], link_target['logical_column_name']) if pd.notna(v)}

# 2. 🚨 강제 패치(Patch): 중복된 'companies ID'를 주체와 객체로 분리
# 물리적 컬럼명이 'COMPANY_ID'와 'ANOTHER_COMPANY_ID'라고 가정
if 'COMPANY_ID' in link_map:
    link_map['COMPANY_ID'] = '고객사 아이디'
if 'ANOTHER_COMPANY_ID' in link_map:
    link_map['ANOTHER_COMPANY_ID'] = '거래처 아이디'

# 3. 데이터에 적용
df_link_raw = pd.read_csv(rf"{path_point}\COMPANY_LINKS.csv", encoding='utf-8-sig')
df_link_raw.columns = [str(col).upper() for col in df_link_raw.columns]
df_link_labeled = df_link_raw.rename(columns=link_map)

# 4. 결과 확인
print("✅ 패치 완료된 링크 데이터 컬럼:", [col for col in df_link_labeled.columns if not col.startswith('_')][:10])

✅ 패치 완료된 링크 데이터 컬럼: ['primary key', '금액', '종류', '고객사 아이디', '생성일자', '삭제일자', '거래일자', '수정일자', '참조 테이블 ID', '참조 테이블명']


In [1]:
import os

# 1. 경로 설정
path_point = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC"
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"

def get_csv_files(directory, label):
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.csv')]
        print(f"\n📂 {label} 폴더 내 파일 목록 (총 {len(files)}개)")
        print("-" * 60)
        for i, f in enumerate(sorted(files)):
            print(f"{i+1:02d}. {f}")
    else:
        print(f"\n❌ 경로를 찾을 수 없습니다: {directory}")

# 2. 파일명 추출 실행
get_csv_files(path_point, "PROD_FLOWPOINT (거래 및 BNPL)")
get_csv_files(path_score, "PROD_FLOWSCORE (기업 및 재무)")


📂 PROD_FLOWPOINT (거래 및 BNPL) 폴더 내 파일 목록 (총 130개)
------------------------------------------------------------
01. ACCOUNTS.csv
02. ASSIGNMENTS.csv
03. ASSIGNMENT_ADD_DOCS.csv
04. ASSIGNMENT_ASSIGNMENT_ADD_DOCS.csv
05. ASSIGNMENT_ASSIGNMENT_DETAILS.csv
06. ASSIGNMENT_DETAILS.csv
07. ASSIGNMENT_DETAIL_PAYABLE_DETAILS.csv
08. ASSIGNMENT_DETAIL_PAYABLE_TRANSACTIONS.csv
09. ASSIGNMENT_DETAIL_RECEIVABLE_DETAILS.csv
10. ASSIGNMENT_DETAIL_RECEIVABLE_TRANSACTIONS.csv
11. ASSIGNMENT_DETAIL_TRANSFER_DETAILS.csv
12. ASSIGNMENT_PAYABLES.csv
13. ASSIGNMENT_RECEIVABLES.csv
14. ASSIGNMENT_TRANSFERS.csv
15. ASSIGNMENT_TRANSFER_CONTRACTS.csv
16. AUTH_CODE_EMAILS.csv
17. AUTH_EMAILS.csv
18. BATCH_JOB_EXECUTION.csv
19. BATCH_JOB_EXECUTION_CONTEXT.csv
20. BATCH_JOB_EXECUTION_PARAMS.csv
21. BATCH_JOB_INSTANCE.csv
22. BATCH_STEP_EXECUTION.csv
23. BATCH_STEP_EXECUTION_CONTEXT.csv
24. BNPL_BUYERS.csv
25. BNPL_BUYER_UPLOAD_FILES.csv
26. BNPL_COMMENTS.csv
27. BNPL_CUSTOMERS.csv
28. BNPL_CUSTOMER_UPLOAD_FILES.cs

In [ ]:
import pandas as pd

# 1. 분석 대상 신규 파일 경로
path_point = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC"
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"

# 2. 샘플링 함수 (컬럼명과 데이터 타입을 한눈에 확인)
def inspect_file(file_path, label):
    print(f"\n🔍 [{label}] 구조 분석")
    print("-" * 50)
    df = pd.read_csv(file_path, encoding='utf-8-sig', nrows=5)
    print(f"컬럼 리스트: {df.columns.tolist()}")
    display(df.head(2))

# 3. 핵심 파일 정찰 실행
inspect_file(rf"{path_point}\BNPL_REQUESTS.csv", "BNPL 신청 정보")
inspect_file(rf"{path_score}\COMPANY_EMPLOYEE_STATISTICS.csv", "인력 통계 상세")


🔍 [BNPL 신청 정보] 구조 분석
--------------------------------------------------
컬럼 리스트: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'UUID', 'USER_ID', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'HOLD_REASON', 'REJECT_REASON', 'EVALUATION_STAGE', 'APPLICATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,UUID,USER_ID,CREATED_AT,DELETED_AT,UPDATED_AT,_AB_CDC_LSN,HOLD_REASON,REJECT_REASON,EVALUATION_STAGE,APPLICATION_STAGE,EVALUATION_STATUS,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS
0,9847dec7-59a6-42a2-814c-601c7084b43e,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,f0ad2ade-945b-46e8-b231-fae54802f670,6,2024-03-08 18:30:04.217000,NaN,2024-03-13 18:36:05.304000,2.869635e+13,NaN,NaN,evaluation,complete,pending,NaN,2025-10-18T02:48:35.772833664Z,complete
1,14d786e3-3fd4-4fca-b395-4f22835b71e0,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,b21b49d4-bf39-44bb-a477-dcb5043f4d45,7,2024-03-11 17:33:37.698000,NaN,2024-04-03 17:37:25.939000,2.869635e+13,NaN,NaN,evaluation,complete,pending,NaN,2025-10-18T02:48:35.772833664Z,complete



🔍 [인력 통계 상세] 구조 분석
--------------------------------------------------
컬럼 리스트: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', 'STANDARD_DATE', 'EMPLOYEE_COUNT', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,STANDARD_DATE,EMPLOYEE_COUNT,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT
0,3c032930-6fb3-4ba4-826a-090c41072454,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,1,2024-12-17 09:29:15.733439,93886392.0,2022-09-13,8,NaN,2025-10-14T11:26:32.305225016Z
1,201e5b01-3257-47e4-af6d-0b17b22fe9b9,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,1,2024-12-17 09:29:15.733439,93886392.0,2020-05-07,4,NaN,2025-10-14T11:26:32.305225016Z


In [ ]:
import pandas as pd

# 1. 추가 분석 대상 파일 경로
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"

extra_targets = {
    "자본_상태": rf"{path_score}\COMPANY_CAPITAL_STATUS.csv",
    "연체_이력": rf"{path_score}\COMPANY_OVERDUE.csv",
    "대표자_이력": rf"{path_score}\COMPANY_REPRESENTATIVE_HISTORY.csv"
}

def quick_inspect(file_path, label):
    print(f"\n" + "="*60)
    print(f"🧐 [{label}] 매뉴얼 매칭성 확인 (상위 3행)")
    print("="*60)
    
    if os.path.exists(file_path):
        # 전체 컬럼명과 샘플 데이터 확인
        df = pd.read_csv(file_path, encoding='utf-8-sig', nrows=3)
        print(f"▶ 컬럼 리스트: {df.columns.tolist()}")
        display(df)
    else:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")

# 실행
for label, p in extra_targets.items():
    quick_inspect(p, label)


🧐 [자본_상태] 매뉴얼 매칭성 확인 (상위 3행)
▶ 컬럼 리스트: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'T_AM', 'DP_AM', 'CHG_DT', 'REG_DT', 'BASE_DT', 'CSTK_CN', 'PSTK_CN', 'COMPANY_ID', 'CREATED_AT', 'ETC_STK_CN', '_AB_CDC_LSN', 'ISS_STK_T_CN', 'W_ISS_STK_T_CN', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,T_AM,DP_AM,CHG_DT,REG_DT,BASE_DT,CSTK_CN,PSTK_CN,COMPANY_ID,CREATED_AT,ETC_STK_CN,_AB_CDC_LSN,ISS_STK_T_CN,W_ISS_STK_T_CN,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT
0,0dfdcad1-9d47-4de3-a1a3-2ac4aaa59cf5,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,NaN,5000,2008-01-30,2008-01-30,2024-11-08,15600000,0,15,2025-01-02 05:49:32.678557,0,93886392.0,15600000,NaN,NaN,2025-10-14T11:26:32.305225016Z
1,92470a1c-eae5-42ca-b195-a2ae1ebeaf01,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,NaN,500,2023-10-26,2023-11-02,2023-12-30,200000,4440,1,2025-01-02 07:59:28.864098,0,93886392.0,211099,NaN,NaN,2025-10-14T11:26:32.305225016Z
2,4e5c9d3a-914c-4eb5-aba0-f0acbb01fc3e,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,3,NaN,5000,2012-07-02,2012-07-02,2024-11-14,261825475,0,36,2025-01-02 08:15:15.160341,0,93886392.0,261825475,NaN,NaN,2025-10-14T11:26:32.305225016Z



🧐 [연체_이력] 매뉴얼 매칭성 확인 (상위 3행)
▶ 컬럼 리스트: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', 'BOND_NUMBER', 'DEBTOR_NAME', 'UPDATE_CODE', 'RELEASE_DATE', 'CREDITOR_NAME', 'DELETION_DATE', 'OVERDUE_AMOUNT', 'REGISTRATION_DATE', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'DEBTOR_COMPANY_NAME', 'RELEASE_REASON_CODE', 'DELETION_REASON_CODE', 'CREDITOR_BUSINESS_NUMBER', 'REGISTRATION_REASON_CODE', 'DEBTOR_CLASSIFICATION_CODE', 'REGISTRATION_REASON_OCCURRENCE_DATE']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,BOND_NUMBER,DEBTOR_NAME,...,REGISTRATION_DATE,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,DEBTOR_COMPANY_NAME,RELEASE_REASON_CODE,DELETION_REASON_CODE,CREDITOR_BUSINESS_NUMBER,REGISTRATION_REASON_CODE,DEBTOR_CLASSIFICATION_CODE,REGISTRATION_REASON_OCCURRENCE_DATE
0,e39a9a50-c610-49e7-862f-b8904098b6e4,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,25,73,2025-01-22 15:13:19.230170,93886392.0,99862,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2021-11-09
1,842830fa-d64c-4207-a192-db80b6667bd0,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,27,73,2025-01-22 15:13:19.233167,93886392.0,99840,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2022-03-13
2,3f708280-b41b-4c05-8e55-19d0352ee4b0,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,29,73,2025-01-22 15:13:19.235450,93886392.0,99834,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2021-11-09



🧐 [대표자_이력] 매뉴얼 매칭성 확인 (상위 3행)
▶ 컬럼 리스트: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'REPRESENTATIVE_HISTORY_INFO']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,REPRESENTATIVE_HISTORY_INFO
0,72b774f8-5cb2-47e7-af3c-d800fa44e141,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,1,2024-12-17 09:29:15.641610,93886392.0,NaN,2025-10-14T11:26:32.305225016Z,"[{""name"": ""박진완"", ""title"": ""사내이사"", ""sequence"": ..."
1,8d11fdbc-6cd8-472d-b6c6-04fc48f8305c,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,7,2024-12-20 07:18:40.674974,93886392.0,NaN,2025-10-14T11:26:32.305225016Z,"[{""name"": ""김유훈"", ""title"": ""대표이사"", ""sequence"": ..."
2,93e101c0-bc8d-4222-9b50-6795d79a08ff,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,3,13,2024-12-20 07:52:08.386839,93886392.0,NaN,2025-10-14T11:26:32.305225016Z,"[{""name"": ""김한준"", ""title"": ""대표이사"", ""sequence"": ..."


In [ ]:
import pandas as pd
import json

# 1. 데이터 로드
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"
df_rep_raw = pd.read_csv(rf"{path_score}\COMPANY_REPRESENTATIVE_HISTORY.csv", encoding='utf-8-sig')

# 2. 대표자 이력 JSON 파싱 함수
def count_rep_changes(json_str):
    try:
        if pd.isna(json_str): return 0
        history = json.loads(json_str)
        return len(history)
    except:
        return 0

# 3. 변경 횟수 컬럼 생성
df_rep_raw['대표자_변경_횟수'] = df_rep_raw['REPRESENTATIVE_HISTORY_INFO'].apply(count_rep_changes)

# 4. 매뉴얼 기준(3회 이상) 위반 기업 추출
risky_rep_firms = df_rep_raw[df_rep_raw['대표자_변경_횟수'] >= 3]

print(f"=== ⚠️ 매뉴얼 기준(변경 3회 이상) 주의 기업: {len(risky_rep_firms)}개사 ===")
display(risky_rep_firms[['COMPANY_ID', '대표자_변경_횟수']].head())

=== ⚠️ 매뉴얼 기준(변경 3회 이상) 주의 기업: 10개사 ===


,COMPANY_ID,대표자_변경_횟수
57,71,3
76,89,3
102,116,3
159,89,3
217,71,3


In [ ]:
import pandas as pd
import os

# 1. 대상 파일 및 경로 정의
path_point = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC"
path_score = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC"

top_targets = {
    "BNPL_신청": rf"{path_point}\BNPL_REQUESTS.csv",
    "인력_통계": rf"{path_score}\COMPANY_EMPLOYEE_STATISTICS.csv",
    "자본_상태": rf"{path_score}\COMPANY_CAPITAL_STATUS.csv",
    "CSS_기존정보": rf"{path_point}\BUSINESS_CSS_INFOS.csv",
    "BNPL_이력뷰": rf"{path_point}\VIEW_PAY_BNPL_REQUEST_HISTORY.csv"
}

def full_target_inspect(targets):
    for label, p in targets.items():
        print(f"\n" + "="*70)
        print(f"🧐 [{label}] 데이터 실체 및 매뉴얼 매칭성 검증")
        print("="*70)
        
        if os.path.exists(p):
            # 컬럼 리스트와 상위 2행 추출
            df = pd.read_csv(p, encoding='utf-8-sig', nrows=2)
            print(f"▶ 컬럼 목록: {df.columns.tolist()}")
            display(df)
        else:
            print(f"⚠️ 파일이 아직 경로에 존재하지 않습니다: {p}")

# 2. 전수 스캔 실행
full_target_inspect(top_targets)


🧐 [BNPL_신청] 데이터 실체 및 매뉴얼 매칭성 검증
▶ 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'UUID', 'USER_ID', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'HOLD_REASON', 'REJECT_REASON', 'EVALUATION_STAGE', 'APPLICATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,UUID,USER_ID,CREATED_AT,DELETED_AT,UPDATED_AT,_AB_CDC_LSN,HOLD_REASON,REJECT_REASON,EVALUATION_STAGE,APPLICATION_STAGE,EVALUATION_STATUS,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS
0,9847dec7-59a6-42a2-814c-601c7084b43e,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,f0ad2ade-945b-46e8-b231-fae54802f670,6,2024-03-08 18:30:04.217000,NaN,2024-03-13 18:36:05.304000,2.869635e+13,NaN,NaN,evaluation,complete,pending,NaN,2025-10-18T02:48:35.772833664Z,complete
1,14d786e3-3fd4-4fca-b395-4f22835b71e0,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,b21b49d4-bf39-44bb-a477-dcb5043f4d45,7,2024-03-11 17:33:37.698000,NaN,2024-04-03 17:37:25.939000,2.869635e+13,NaN,NaN,evaluation,complete,pending,NaN,2025-10-18T02:48:35.772833664Z,complete



🧐 [인력_통계] 데이터 실체 및 매뉴얼 매칭성 검증
▶ 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', 'STANDARD_DATE', 'EMPLOYEE_COUNT', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,STANDARD_DATE,EMPLOYEE_COUNT,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT
0,3c032930-6fb3-4ba4-826a-090c41072454,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,1,2024-12-17 09:29:15.733439,93886392.0,2022-09-13,8,NaN,2025-10-14T11:26:32.305225016Z
1,201e5b01-3257-47e4-af6d-0b17b22fe9b9,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,1,2024-12-17 09:29:15.733439,93886392.0,2020-05-07,4,NaN,2025-10-14T11:26:32.305225016Z



🧐 [자본_상태] 데이터 실체 및 매뉴얼 매칭성 검증
▶ 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'T_AM', 'DP_AM', 'CHG_DT', 'REG_DT', 'BASE_DT', 'CSTK_CN', 'PSTK_CN', 'COMPANY_ID', 'CREATED_AT', 'ETC_STK_CN', '_AB_CDC_LSN', 'ISS_STK_T_CN', 'W_ISS_STK_T_CN', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,T_AM,DP_AM,CHG_DT,REG_DT,BASE_DT,CSTK_CN,PSTK_CN,COMPANY_ID,CREATED_AT,ETC_STK_CN,_AB_CDC_LSN,ISS_STK_T_CN,W_ISS_STK_T_CN,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT
0,0dfdcad1-9d47-4de3-a1a3-2ac4aaa59cf5,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,NaN,5000,2008-01-30,2008-01-30,2024-11-08,15600000,0,15,2025-01-02 05:49:32.678557,0,93886392.0,15600000,NaN,NaN,2025-10-14T11:26:32.305225016Z
1,92470a1c-eae5-42ca-b195-a2ae1ebeaf01,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,NaN,500,2023-10-26,2023-11-02,2023-12-30,200000,4440,1,2025-01-02 07:59:28.864098,0,93886392.0,211099,NaN,NaN,2025-10-14T11:26:32.305225016Z



🧐 [CSS_기존정보] 데이터 실체 및 매뉴얼 매칭성 검증
▶ 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'YEAR', 'TYPE_NAME', 'COMPANY_ID', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'TOTAL_SCORE', 'COMPANY_NAME', 'ENROLLMENT_DATE', 'BUSINESS_USER_ID', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'BUSINESS_REGISTRATION_NUMBER']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,YEAR,TYPE_NAME,COMPANY_ID,CREATED_AT,DELETED_AT,UPDATED_AT,_AB_CDC_LSN,TOTAL_SCORE,COMPANY_NAME,ENROLLMENT_DATE,BUSINESS_USER_ID,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,BUSINESS_REGISTRATION_NUMBER
0,9665a62d-21c6-49df-a17d-099a897af880,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,2020,전문설비시공,94,2022-10-26 17:58:59.183000,NaN,NaN,2.869635e+13,70.00,한솔이엠이,2022-10-26,NaN,NaN,2025-10-18T02:48:35.772833664Z,3148144231
1,822ac9e5-e6cb-4b99-9d5d-4f0c33620789,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,2020,건설업,76,2022-10-26 17:58:59.183000,NaN,NaN,2.869635e+13,128.55,일성건설,2022-10-26,NaN,NaN,2025-10-18T02:48:35.772833664Z,1058129640



🧐 [BNPL_이력뷰] 데이터 실체 및 매뉴얼 매칭성 검증
▶ 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'TYPE', 'IS_READ', 'ITEM_ID', 'COMPANY_ID', 'CREATED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'IS_INTERNAL', 'COMMENT_UUID', 'ADMIN_USER_ID', 'COMMENT_CONTENT', 'EVALUATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS', 'PAY_BNPL_REQUEST_ID']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,TYPE,IS_READ,ITEM_ID,COMPANY_ID,CREATED_AT,UPDATED_AT,...,IS_INTERNAL,COMMENT_UUID,ADMIN_USER_ID,COMMENT_CONTENT,EVALUATION_STAGE,EVALUATION_STATUS,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS,PAY_BNPL_REQUEST_ID
0,025182d9-252c-4316-9be2-e77c74356607,2026-02-24 14:11:47.278000+00:00,"{\n ""changes"": [],\n ""sync_id"": 788\n}",136,EVALUATION_LOG,NaN,324,NaN,2025-08-26 09:52:24.052000,NaN,...,NaN,NaN,5.0,NaN,NEGOTIATING,COMPLETE_EVALUATION,NaN,NaN,NaN,304
1,f8a150ee-fbf4-41c0-bb0e-3f2564a99864,2026-02-24 14:11:47.278000+00:00,"{\n ""changes"": [],\n ""sync_id"": 788\n}",136,EVALUATION_LOG,NaN,468,NaN,2025-11-28 09:30:31.952000,NaN,...,NaN,NaN,NaN,NaN,REVIEWING_DOCUMENTS,EVALUATING,NaN,NaN,COMPLETED,424


In [2]:
import os

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
paths = [
    os.path.join(base_path, "PROD_FLOWSCORE", "PUBLIC"),
    os.path.join(base_path, "PROD_FLOWPOINT", "PUBLIC")
]

for p in paths:
    print(f"\n📂 경로: {p}")
    if os.path.exists(p):
        print(os.listdir(p))
    else:
        print("❌ 경로가 존재하지 않습니다.")


📂 경로: C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWSCORE\PUBLIC
['BUSINESS.csv', 'COMPANY.csv', 'COMPANY_CAPITAL_STATUS.csv', 'COMPANY_CASH_FLOW.csv', 'COMPANY_CREDIT_GRADE.csv', 'COMPANY_EMPLOYEE_STATISTICS.csv', 'COMPANY_EXECUTIVES.csv', 'COMPANY_EXECUTIVE_DETAIL.csv', 'COMPANY_FINANCIAL_RATIO.csv', 'COMPANY_FINANCIAL_STATEMENT.csv', 'COMPANY_HISTORY.csv', 'COMPANY_INCOME_STATEMENT.csv', 'COMPANY_KOREAN_CODE_V1.csv', 'COMPANY_MORATORIUM.csv', 'COMPANY_OVERDUE.csv', 'COMPANY_OVERVIEW.csv', 'COMPANY_PENSION_EMPLOYEE_STATISTICS.csv', 'COMPANY_REPRESENTATIVE.csv', 'COMPANY_REPRESENTATIVE_HISTORY.csv', 'COMPANY_SEARCH_KOREAN_CODE_V1.csv', 'COMPANY_STOCKHOLDER.csv', 'EWA_CODE_V1.csv', 'EWA_CODE_V2.csv', 'EWA_GRADE.csv', 'FINANCIAL_STATUS.csv', 'PRIVILEGE.csv', 'ROLES.csv', 'SEARCH_COMPANY_HISTORY.csv', 'USERS.csv', 'USER_COMPANY_SEARCH_HISTORY.csv']

📂 경로: C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC
['ACCOUNTS.csv', 'ASSIGNMENTS.csv', 'ASSIGNM